# Подготовка датасета для экспериментов

| Файл | Зачем нужен |
|---|---|
| `data/val_dataset.json` | Ссылки на фото и группы |
| `data/golden.jsonl` | Эталонные рамки для расчёта качества |
| `results/predictions/grounding_dino.jsonl` | Готовые рамки Grounding DINO для эксперимента с удалением дублей |

Объединяем записи по `photo_id` в один `butterfly_validation.json`. Это единственный файл данных, который нужно загрузить в Kaggle или Colab. В этом ноутбуке модели не запускаются.


In [1]:
import json
from pathlib import Path

PHOTOS_PATH = Path("data/val_dataset.json")
GOLDEN_PATH = Path("data/golden.jsonl")
DINO_PATH = Path("results/predictions/grounding_dino.jsonl")
OUTPUT_PATH = Path("butterfly_validation.json")

for path in (PHOTOS_PATH, GOLDEN_PATH, DINO_PATH):
    if not path.is_file():
        raise FileNotFoundError(
            f"Не найден {path}. Текущая папка: {Path.cwd()}. "
            "Запусти ноутбук из корня репозитория или поправь путь в этой ячейке."
        )


## Читаем файлы и сверяем ID

В JSON фото хранятся одним списком. В JSONL каждая строка — отдельная запись. У всех трёх файлов должны совпадать наборы из 400 уникальных ID. Порядок строк может различаться.


In [2]:
def read_jsonl(path):
    with path.open(encoding="utf-8-sig") as file:
        return [json.loads(line) for line in file if line.strip()]


def index_by_photo_id(rows, path, required_fields):
    if not isinstance(rows, list):
        raise ValueError(f"{path}: ожидался список записей")
    indexed = {}
    for i, row in enumerate(rows, 1):
        missing = required_fields - row.keys()
        if missing:
            raise ValueError(f"{path}, запись {i}: отсутствуют поля {sorted(missing)}")
        photo_id = row["photo_id"]
        if type(photo_id) is not int or photo_id <= 0 or photo_id in indexed:
            raise ValueError(f"{path}, запись {i}: некорректный или повторный photo_id {photo_id!r}")
        indexed[photo_id] = row
    return indexed


photos = index_by_photo_id(
    json.loads(PHOTOS_PATH.read_text(encoding="utf-8-sig")),
    PHOTOS_PATH, {"photo_id", "url", "group"},
)
golden = index_by_photo_id(
    read_jsonl(GOLDEN_PATH), GOLDEN_PATH, {"photo_id", "boxes"},
)
dino = index_by_photo_id(
    read_jsonl(DINO_PATH), DINO_PATH, {"photo_id", "boxes", "error"},
)

if len(photos) != 400:
    raise ValueError(f"{PHOTOS_PATH}: ожидалось 400 фото, найдено {len(photos)}")
for path, rows in ((GOLDEN_PATH, golden), (DINO_PATH, dino)):
    missing = photos.keys() - rows.keys()
    extra = rows.keys() - photos.keys()
    if missing or extra:
        raise ValueError(
            f"{path}: отсутствует ID {len(missing)}, например {sorted(missing)[:3]}. "
            f"Лишних ID {len(extra)}, например {sorted(extra)[:3]}."
        )
print("Во всех трёх файлах совпали 400 photo_id.")


Во всех трёх файлах совпали 400 photo_id.


## Записываем файл для Kaggle и Colab

`boxes` на верхнем уровне — эталон. `grounding_dino.boxes` — рамки, которые предсказала модель. Сохраняем также `grounding_dino.error`, чтобы при оценке учитывать технические сбои.


In [3]:
combined = [
    {
        "photo_id": photo_id,
        "url": photo["url"],
        "group": photo["group"],
        "boxes": golden[photo_id]["boxes"],
        "grounding_dino": {
            "boxes": dino[photo_id]["boxes"],
            "error": dino[photo_id]["error"],
        },
    }
    for photo_id, photo in photos.items()
]

OUTPUT_PATH.write_text(
    json.dumps(combined, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)
print(f"Сохранено {len(combined)} фото: {OUTPUT_PATH.resolve()}")


Сохранено 400 фото: /home/daryas/data-labeling-shad/butterfly_validation.json
